In [1]:
import numpy as np
from random import *
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import csv

# Criterions

Depending on the type of problem (classification or regression) and the quanttiative criteria used to split nodes, we may need different expressions. In order to make the code clear and avoid writing it twice we regroup everything related to node splitting and evaluation of the leaves in one class. 

In [3]:
class Criterion:
    """
    Class to store specificities due to the type of tree (regression/classificaion) and to the criterion used
    entrop, variance, gini index...
    """
    def initialize(self, ySorted):
        pass
    def moveSample(self, y):
        pass
    def score(self):
        pass
    def output(self, ySorted):
        pass
  
class VarianceCriterion(Criterion):
    def initialize(self, ySorted):
        """
        Initialize the saved values to compute the variance of the left and right nodes after a split
        """
        self.N = len(ySorted)
        self.leftN = 0 
        self.leftSum = 0 
        self.leftSum2 = 0

        self.rightN = self.N
        self.rightSum = np.sum(np.array(ySorted))
        self.rightSum2 = np.sum(np.array(ySorted)**2)
    
    def moveSample(self, y):
        """
        Update the saved values knowing that the y eleement has been moved from the right node to the left node
        """
        self.leftN += 1
        self.rightN -= 1

        self.leftSum += y
        self.rightSum -= y

        self.leftSum2 += y**2
        self.rightSum2 -= y**2
        
    def score(self):
        """
        compute the variance of the left and right nodes after a split and return the weighted average of these variances
        """
        if self.N == 0:
            return 0
        varianceLeft = self.leftSum2/self.leftN  - (self.leftSum/self.leftN)**2 
        varianceRight = self.rightSum2/self.rightN  - (self.rightSum/self.rightN)**2 
        return -(varianceLeft*self.leftN + varianceRight*self.rightN)/self.N
    
    def output(self, ySorted):
        """
        Return the mean of the y values in the node as output for regression trees
        """
        return np.mean(ySorted)
    
class EntropyCriterion(Criterion):
    def initialize(self, ySorted):
        self.N = len(ySorted)

        classes = set(ySorted)
        self.mapping = {val: i for i, val in enumerate(classes)}

        self.leftCount = 0
        self.leftNumbers = np.zeros(len(classes))

        self.rightCount = self.N
        self.rightNumbers = np.array([np.sum(ySorted == c) for c in classes])
    
    def moveSample(self, y):
        self.leftCount += 1
        self.rightCount -= 1

        self.leftNumbers[self.mapping[y]] += 1
        self.rightNumbers[self.mapping[y]] -= 1
        
    def score(self):
        entropyLeft = entropy(self.leftNumbers/self.leftCount)
        entropyRight = entropy(self.rightNumbers/self.rightCount)
        return (entropyLeft*self.leftCount + entropyRight*self.rightCount)/self.N
    
    def output(self, ySorted):
        counts = np.bincount(ySorted.astype(int))
        return np.argmax(counts)

def entropy(probabilities):
  return np.sum(probabilities * np.log2(probabilities + 1e-10))


# Node structure

Nodes will contain:
-  the score of the split they perform (if they are not a leaf)
-  their depth and 
-  the number of samples at that point of the tree
-  the index chosen to cut and its corresponding cutting threshold
-  the output if this is a leaf

In [6]:
class Node :
    def __init__(self):
        self.score = None

        self.nSamples = None
        self.depth = None

        self.featureIndex = None
        self.featureThreshold = None
        
        self.left = None
        self.right = None

        self.output = None

    def __str__(self):
        return self._to_string()
    
    def _to_string(self, espace=0):
        indent = " " * espace

        if self.left is None and self.right is None:
            return f"{indent}[{self.output:.1f}]\n"

        text = f"{indent}[X{self.featureIndex} {self.featureThreshold:.1f}]\n"

        if self.left is not None:
            text += self.left._to_string(espace + 2)
        if self.right is not None:
            text += self.right._to_string(espace + 2)

        return text

# Tree strucrure

In [37]:
class DecisionTree :
    def __init__(self, criterion, maxDepth=10, minSamplesSplit=2, minSamplesLeaf=1):
        self.max_depth = maxDepth
        self.minSamplesSplit = minSamplesSplit
        self.minSamplesLeaf = minSamplesLeaf

        self.root = None

        self.criterion = criterion
        
        self.tree = None

    def best_split(self, X, y, numberOfArgs):
        N = len(y)

        bestIndex , bestThreshold, bestScore = None, None, -1000
        attributs = sample([k for k in range(len(X[0]))], numberOfArgs)
        for index in attributs :
            #Sort by the current attribute
            seuils, classes = zip(*sorted(zip(X[:, index], y)))
            self.criterion.initialize(classes)
            
            for i in range(self.minSamplesLeaf, N-self.minSamplesLeaf):
                self.criterion.moveSample(classes[i-1])
                if seuils[i] == seuils[i-1]:
                    continue
                
                score = self.criterion.score()
                if score > bestScore :
                    bestScore = score
                    bestIndex = index
                    bestThreshold = (seuils[i] + seuils[i-1]) / 2 
        return bestIndex, bestThreshold, bestScore

    def planting(self, X, y, numberOfArgs,  depth=0):
        m = y.size
        node = Node()

        node.depth = depth
        node.nSamples = m
        
        def makeLeaf():
            node.output = self.criterion.output(y)
            return node
        if m <= self.minSamplesSplit or depth > self.max_depth:
            return makeLeaf()

        index, seuil, score = self.best_split(X, y, numberOfArgs)
        if index == None :
            return makeLeaf()
        indexLeft = X[:, index] < seuil
        XLeft, yLeft = X[indexLeft] , y[indexLeft]
        XRight, yRight = X[np.logical_not(indexLeft)], y[np.logical_not(indexLeft)]
        
        if len(yLeft) == 0 or len(yRight) == 0:
            return makeLeaf()
        
        node.featureIndex = index
        node.featureThreshold = seuil

        node.left = self.planting(XLeft, yLeft, numberOfArgs,  depth + 1)
        node.right = self.planting(XRight, yRight, numberOfArgs,  depth + 1)
        node.score = score   
        return node

    def fit(self, X, y, numberOfArgs):
        self.tree = self.planting(X, y, numberOfArgs)
        
    def prediction(self, input):
        
        node = self.tree
        while node.left!= None :
            if input[node.featureIndex] < node.featureThreshold  :
                node = node.left
            else :
                node = node.right
        return node.output

    def predictions(self, X):
        return [self.prediction(entrees) for entrees in X]
    
    def __str__(self):
        return self.tree.__str__()
    


# Random forest

In [43]:
class RandomForest :
    def __init__(self, nTrees, criterion, maxDepth=10, minSampleSplit=2, minSamplesLeaf=1, numberOfArgs = 6):
        self.nTrees = nTrees
        self.trees = [DecisionTree(criterion, maxDepth, minSampleSplit, minSamplesLeaf) for _ in range(nTrees)]
        self.numberOfArgs = numberOfArgs
    
    def bagging(self, data):
        samples = []
        n = len(data)
        for _ in range(self.nTrees): 
            tempBag = []
            for _ in range(n):
                tempBag += [data[randrange(0, n)]]
            samples += [tempBag]
        return samples
    
    def make(self, data):
        samples = self.bagging(data)
        for idx, sample in enumerate(samples):
            self.trees[idx].fit(np.array(sample)[:, :-1], np.array(sample)[:, -1], self.numberOfArgs)
        
    def predict(self, inputs) :
        predit = []
        for input in inputs :
            predit += [1/self.nTrees* np.sum([tree.prediction(input) for tree in self.trees])]
        return predit
        

In [45]:
def predictor(data, numerOfTrees, numberOfArgs, criterion = VarianceCriterion(), maxDepth = 8, minSampleSplit = 2, minSamplesLeaf = 1,):

    train, test = train_test_split(data, test_size = 0.25)

    forest = RandomForest(numerOfTrees, criterion, 
                          maxDepth=maxDepth, minSampleSplit=minSampleSplit,
                          minSamplesLeaf=minSamplesLeaf, 
                          numberOfArgs=numberOfArgs)
    forest.make(train)
    
    erreurTrain = erreurTest = 0
    L1 = forest.predict(np.array(test)[:, :-1])
    L2 = forest.predict(np.array(train)[:, :-1])

    for k in range(len(L1)) :
        erreurTest += (L1[k] - test[k][-1])**2
    for k in range(len(L2)) :
        erreurTrain += (L2[k] - train[k][-1])**2
    
    erreurTest = np.sqrt(erreurTest/len(L1))
    erreurTrain = np.sqrt(erreurTrain/len(L2))
    return forest, erreurTest, erreurTrain, L1, test

In [68]:
def readData(file, delta = 2):
    donnees = csv.reader(file, delimiter=',')
    k, X, Y=0, [], []
    for k, ligne in enumerate(donnees):
        if k == 0:
            continue  # skip header
        def f(i):
            return float(ligne[i].replace(",", "."))
        y = f(1)
        #x = [1/f(4)**2, 1/f(7)**2,f(8),f(9), f(10), f(11), abs(f(12)), abs(f(15))]
        x = [f(4), f(7), abs(f(12)), abs(f(15))]
        x = [1/f(4)**2, 1/f(7)**2, f(8), f(9), f(10), f(11), abs(f(12))]
        if k == 1:
            print(y, x)
        X.append(x)
        Y.append(y)

    X, Y = X[:2499-delta], Y[delta:]
    for k in range(len(X)):
        X[k] += [Y[k] - 20]
    return np.array(X)

In [ ]:
import itertools
import json
file = open("data.csv","r")
data = readData(file, delta = 2)
numberOfTrees = 10
numberOfArgs = 7
maxDepth = 8
minSamplesSplit = 2
minSamplesLeaf = 1
criterion = VarianceCriterion()

numberOfTreesToTry = [1, 5, 10, 20, 50, 100]
numberOfArgsToTry = [1, 3, 5, 7]
maxDepthToTry = [2, 4, 6, 8, 10]
results = []

for n_trees, n_args, depth in itertools.product(
    numberOfTreesToTry,
    numberOfArgsToTry,
    maxDepthToTry
):
    avgErrorTest = 0
    avgErrorTrain = 0
    for i in range(5):
        forest, errorTest, errorTrain, L1, test = predictor(
            data,
            n_trees,
            n_args,
            criterion,
            depth,
            minSamplesSplit,
            minSamplesLeaf
        )
        avgErrorTest += errorTest
        avgErrorTrain += errorTrain
    avgErrorTest /= 5
    avgErrorTrain /= 5
    print(f"n_trees: {n_trees}, n_args: {n_args}, max_depth: {depth}, test_error: {avgErrorTest:.4f}, train_error: {avgErrorTrain:.4f}")
    results.append({
        "n_trees": n_trees,
        "n_args": n_args,
        "max_depth": depth,
        "test_error": avgErrorTest,
        "train_error": avgErrorTrain
    })

# trier les meilleurs modèles
results = sorted(results, key=lambda x: x["test_error"])
with open("results.json", "w") as f:
    json.dump(results, f, indent=4)

46.0 [1.0341975818241307, 138197.11978673973, 0.180151403, -0.983638893, -0.727403685, -0.686209792, 0.543939847]
n_trees: 1, n_args: 1, max_depth: 2, test_error: 17.0750, train_error: 16.8688
n_trees: 1, n_args: 1, max_depth: 4, test_error: 15.4387, train_error: 14.8751
n_trees: 1, n_args: 1, max_depth: 6, test_error: 13.9977, train_error: 12.1309
n_trees: 1, n_args: 1, max_depth: 8, test_error: 14.0671, train_error: 11.1129
n_trees: 1, n_args: 1, max_depth: 10, test_error: 13.9575, train_error: 9.9143
n_trees: 1, n_args: 3, max_depth: 2, test_error: 11.1018, train_error: 10.6193
n_trees: 1, n_args: 3, max_depth: 4, test_error: 8.7556, train_error: 7.9049
n_trees: 1, n_args: 3, max_depth: 6, test_error: 7.7861, train_error: 6.3361
n_trees: 1, n_args: 3, max_depth: 8, test_error: 7.6592, train_error: 5.6343
n_trees: 1, n_args: 3, max_depth: 10, test_error: 7.0429, train_error: 4.4207
n_trees: 1, n_args: 5, max_depth: 2, test_error: 9.0872, train_error: 8.8231
n_trees: 1, n_args: 5, max

KeyboardInterrupt: 